In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("Practice DF- 3"). \
getOrCreate()

In [2]:
product_df = spark.read \
.format("csv") \
.option("header", "true") \
.option("inferSchema", "true") \
.load("D:\Learn-spark\learn-spark-maide\product_list.csv")

1. Tạo cột mới với tên là total_price = product_price * quantity

In [3]:
product_df.show()

+----------+-------------------+--------------------+-------------+--------+
|product_id|product_category_id|        product_name|product_price|quantity|
+----------+-------------------+--------------------+-------------+--------+
|         1|                  6|Convertible Roadster|        132.0|      35|
|         2|                  6|          SUV Luxury|         98.0|      27|
|         3|                  8|     Vintage Classic|        157.0|       2|
|         4|                  8|          Tuning Kit|         43.0|      19|
|         5|                 10|  Classic Muscle Car|         55.0|      25|
|         6|                  5|        Sports Coupe|        138.0|      28|
|         7|                  3|    Car Audio System|        142.0|      28|
|         8|                  2|           Cargo Van|         14.0|      21|
|         9|                  5|    Luxury Limousine|         43.0|      30|
|        10|                  2|       Off-road Jeep|         14.0|      49|

Cách 1: dùng rất rất nhiều

In [4]:
from pyspark.sql.functions import *

In [5]:
product_df.withColumn("total_price", col("product_price") * col("quantity")).show()

+----------+-------------------+--------------------+-------------+--------+-----------+
|product_id|product_category_id|        product_name|product_price|quantity|total_price|
+----------+-------------------+--------------------+-------------+--------+-----------+
|         1|                  6|Convertible Roadster|        132.0|      35|     4620.0|
|         2|                  6|          SUV Luxury|         98.0|      27|     2646.0|
|         3|                  8|     Vintage Classic|        157.0|       2|      314.0|
|         4|                  8|          Tuning Kit|         43.0|      19|      817.0|
|         5|                 10|  Classic Muscle Car|         55.0|      25|     1375.0|
|         6|                  5|        Sports Coupe|        138.0|      28|     3864.0|
|         7|                  3|    Car Audio System|        142.0|      28|     3976.0|
|         8|                  2|           Cargo Van|         14.0|      21|      294.0|
|         9|         

Cách 2: Dùng select

In [ ]:
product_df.select("*", expr("product_price * quantity as total_price")).show()
#viết expr nhiều lần

+----------+-------------------+--------------------+-------------+--------+-----------+
|product_id|product_category_id|        product_name|product_price|quantity|total_price|
+----------+-------------------+--------------------+-------------+--------+-----------+
|         1|                  6|Convertible Roadster|        132.0|      35|     4620.0|
|         2|                  6|          SUV Luxury|         98.0|      27|     2646.0|
|         3|                  8|     Vintage Classic|        157.0|       2|      314.0|
|         4|                  8|          Tuning Kit|         43.0|      19|      817.0|
|         5|                 10|  Classic Muscle Car|         55.0|      25|     1375.0|
|         6|                  5|        Sports Coupe|        138.0|      28|     3864.0|
|         7|                  3|    Car Audio System|        142.0|      28|     3976.0|
|         8|                  2|           Cargo Van|         14.0|      21|      294.0|
|         9|         

In [8]:
product_df.selectExpr("*", "product_price * quantity as total_price").show()

+----------+-------------------+--------------------+-------------+--------+-----------+
|product_id|product_category_id|        product_name|product_price|quantity|total_price|
+----------+-------------------+--------------------+-------------+--------+-----------+
|         1|                  6|Convertible Roadster|        132.0|      35|     4620.0|
|         2|                  6|          SUV Luxury|         98.0|      27|     2646.0|
|         3|                  8|     Vintage Classic|        157.0|       2|      314.0|
|         4|                  8|          Tuning Kit|         43.0|      19|      817.0|
|         5|                 10|  Classic Muscle Car|         55.0|      25|     1375.0|
|         6|                  5|        Sports Coupe|        138.0|      28|     3864.0|
|         7|                  3|    Car Audio System|        142.0|      28|     3976.0|
|         8|                  2|           Cargo Van|         14.0|      21|      294.0|
|         9|         

2. Sửa giá trị trong cột product_price:
- Nếu product_name là sản phẩm Luxury thì tăng giá lên 20%
- Nếu product_name là sản phẩm là Vintage thì tăng giá lên 50%
- Còn lại giữ nguyên giá

In [10]:
#Cach 1
product_df.withColumn("product_price",
                      when(col("product_name").contains("Luxury"), col("product_price")*1.2)
                      .when(col("product_name").contains("Vintage"), col("product_price")*1.5)
                      .otherwise(col("product_price"))).show()

+----------+-------------------+--------------------+-------------+--------+
|product_id|product_category_id|        product_name|product_price|quantity|
+----------+-------------------+--------------------+-------------+--------+
|         1|                  6|Convertible Roadster|        132.0|      35|
|         2|                  6|          SUV Luxury|        117.6|      27|
|         3|                  8|     Vintage Classic|        235.5|       2|
|         4|                  8|          Tuning Kit|         43.0|      19|
|         5|                 10|  Classic Muscle Car|         55.0|      25|
|         6|                  5|        Sports Coupe|        138.0|      28|
|         7|                  3|    Car Audio System|        142.0|      28|
|         8|                  2|           Cargo Van|         14.0|      21|
|         9|                  5|    Luxury Limousine|         51.6|      30|
|        10|                  2|       Off-road Jeep|         14.0|      49|

In [12]:
#Cach 2
product_df.selectExpr("*", 
                      """
                    CASE
                        WHEN product_name like '%Luxury%' THEN product_price * 1.2
                        WHEN product_name like '%Vintage%' THEN product_price * 1.5
                        ELSE product_price END new_price

                        """).drop("product_price").withColumnRenamed("new_price", "product_price").show()

+----------+-------------------+--------------------+--------+-------------+
|product_id|product_category_id|        product_name|quantity|product_price|
+----------+-------------------+--------------------+--------+-------------+
|         1|                  6|Convertible Roadster|      35|        132.0|
|         2|                  6|          SUV Luxury|      27|        117.6|
|         3|                  8|     Vintage Classic|       2|        235.5|
|         4|                  8|          Tuning Kit|      19|         43.0|
|         5|                 10|  Classic Muscle Car|      25|         55.0|
|         6|                  5|        Sports Coupe|      28|        138.0|
|         7|                  3|    Car Audio System|      28|        142.0|
|         8|                  2|           Cargo Van|      21|         14.0|
|         9|                  5|    Luxury Limousine|      30|         51.6|
|        10|                  2|       Off-road Jeep|      49|         14.0|